# Flappy Bird Double DQN on Google Colab

Before running the notebook, select **Runtime > Change runtime type > GPU**. Checkpoints and CSV logs are stored in Google Drive so the run can be resumed after a Colab disconnect.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os

REPOSITORY = 'https://github.com/echiu12345/flappy_bird2.git'
PROJECT_DIR = '/content/flappy_bird2'
if not os.path.isdir(os.path.join(PROJECT_DIR, '.git')):
    !git clone {REPOSITORY} {PROJECT_DIR}
%cd {PROJECT_DIR}
!git pull --ff-only
!python -m pip install -q pygame

In [ ]:
import tensorflow as tf

print('TensorFlow:', tf.__version__)
print('GPU devices:', tf.config.list_physical_devices('GPU'))
assert tf.config.list_physical_devices('GPU'), (
    'GPU was not detected. Select Runtime > Change runtime type > GPU, then restart the runtime.'
)

In [ ]:
import os
import shutil

DRIVE_OUTPUT = '/content/drive/MyDrive/flappybird/state_experiments'
LOCAL_OUTPUT = os.path.join(PROJECT_DIR, 'state_experiments')
os.makedirs(DRIVE_OUTPUT, exist_ok=True)
if os.path.lexists(LOCAL_OUTPUT) and not os.path.islink(LOCAL_OUTPUT):
    if os.path.isdir(LOCAL_OUTPUT) and not os.listdir(LOCAL_OUTPUT):
        os.rmdir(LOCAL_OUTPUT)
    else:
        raise RuntimeError(f'Move or remove the existing output first: {LOCAL_OUTPUT}')
if not os.path.lexists(LOCAL_OUTPUT):
    os.symlink(DRIVE_OUTPUT, LOCAL_OUTPUT)
print('Training output:', os.path.realpath(LOCAL_OUTPUT))

## Train or resume DDQN

The same cell starts a new run when no checkpoint exists and resumes it when a checkpoint is present. Do not add `--from-scratch` when resuming.

In [ ]:
RUN_NAME = 'state-ddqn-seed42'
SEED = 42
MAX_STEPS = 500_000

!python state_q_network.py \
    --algorithm ddqn \
    --run-name {RUN_NAME} \
    --seed {SEED} \
    --max-steps {MAX_STEPS} \
    --checkpoint-interval 50

Stop the cell with Colab's stop button when necessary. The script catches a normal interrupt and saves another checkpoint. Colab may terminate a runtime without sending an interrupt, so the periodic checkpoints remain important.

In [ ]:
!python state_q_network.py \
    --mode eval \
    --algorithm ddqn \
    --run-name {RUN_NAME} \
    --seed 1042 \
    --eval-episodes 100

In [ ]:
CSV_PATH = f'state_experiments/{RUN_NAME}/episodes.csv'
GRAPH_PATH = f'state_experiments/{RUN_NAME}/progress.png'
!python plot_comparison.py --ddqn {CSV_PATH} --output {GRAPH_PATH}
from IPython.display import Image, display
display(Image(filename=GRAPH_PATH))